# 🛰️ Debris-Scan AI — Backend Server

This notebook runs the entire FastAPI backend for the Debris-Scan AI mission control system.

**Instructions:**
1. Click **Runtime → Run all** (or Ctrl+F9)
2. Wait for the ngrok URL to appear in the last cell's output
3. Copy that URL into the frontend's backend connector field
4. Create a mission and start the pipeline from the frontend

> ⚠️ **Simulated Mission Mode**: All sensor data is synthetic. The same pipeline architecture runs unchanged when real drone hardware is connected via the `LiveScanAdapter`.

---

## 1. Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn pyngrok websockets python-multipart scipy numpy aiofiles

## 2. Data Models (Pydantic Schemas)

In [ ]:
%%writefile models.py
"""Debris-Scan AI — Pydantic Data Models"""
from pydantic import BaseModel, Field
from enum import Enum
from datetime import datetime, timezone
from typing import Optional, Any
import uuid

class StageEnum(str, Enum):
    INIT        = "init"
    SCAN        = "scan"
    RECONSTRUCT = "reconstruct"
    DETECT      = "detect"
    FUSE        = "fuse"
    REVISIT     = "revisit"
    HAZARD      = "hazard"
    ROUTE       = "route"
    REPORT      = "report"

STAGE_ORDER = [
    StageEnum.SCAN, StageEnum.RECONSTRUCT, StageEnum.DETECT,
    StageEnum.FUSE, StageEnum.REVISIT, StageEnum.HAZARD,
    StageEnum.ROUTE, StageEnum.REPORT,
]

class StatusEnum(str, Enum):
    PENDING  = "pending"
    STARTED  = "started"
    PROGRESS = "progress"
    COMPLETE = "complete"
    FAILED   = "failed"

class StageEvent(BaseModel):
    mission_id: str
    stage: StageEnum
    status: StatusEnum
    progress: Optional[float] = None
    payload: Optional[dict[str, Any]] = None
    timestamp: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

class MissionCreate(BaseModel):
    site_name: str = "Sector Alpha — Collapsed Commercial Complex"
    boundary_coords: list[list[float]] = Field(
        default=[[28.6139, 77.2090], [28.6145, 77.2090],
                 [28.6145, 77.2096], [28.6139, 77.2096]])

class Mission(BaseModel):
    id: str = Field(default_factory=lambda: str(uuid.uuid4())[:8])
    site_name: str
    boundary_coords: list[list[float]]
    current_stage: StageEnum = StageEnum.INIT
    stage_statuses: dict[str, str] = Field(default_factory=dict)
    artifacts: dict[str, Any] = Field(default_factory=dict)
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    started_at: Optional[datetime] = None
    completed_at: Optional[datetime] = None
    adapter_mode: str = "mock"

class Detection(BaseModel):
    id: str
    source: str
    x: float
    y: float
    z: float = 0.0
    confidence: float
    label: str = "person"
    metadata: dict[str, Any] = Field(default_factory=dict)

class FusionResult(BaseModel):
    id: str
    x: float
    y: float
    z: float
    total_score: float
    thermal_contribution: float
    radar_contribution: float
    structural_contribution: float
    hazard_penalty: float
    label: str = "survivor_candidate"
    confirmed: bool = False

class HazardCell(BaseModel):
    x: int
    y: int
    cost: float
    classification: str = "stable"

class Waypoint(BaseModel):
    x: float
    y: float
    z: float = 0.0
    hazard_cost: float = 0.0

class RouteResult(BaseModel):
    target_id: str
    mode: str
    waypoints: list[Waypoint]
    total_distance_m: float
    estimated_time_min: float
    max_hazard_encountered: float
    hazard_zone_crossings: int

class MissionReport(BaseModel):
    mission_id: str
    site_name: str
    adapter_mode: str
    total_area_scanned_m2: float
    scan_duration_s: float
    total_points_captured: int
    candidates_detected: int
    confirmed_survivors: int
    fusion_results: list[FusionResult]
    routes: list[RouteResult]
    hazard_summary: dict[str, int] = Field(default_factory=dict)
    generated_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

print("✅ Models loaded")

## 3. Sensor Adapters (Mock + Live stubs)

In [ ]:
%%writefile adapters.py
"""Debris-Scan AI — Sensor Adapters"""
import numpy as np
import asyncio
from typing import AsyncIterator, Protocol, runtime_checkable
from dataclasses import dataclass, field

@dataclass
class Frame:
    index: int
    data: np.ndarray
    timestamp: float
    width: int = 0
    height: int = 0
    def __post_init__(self):
        if self.data is not None:
            self.height, self.width = self.data.shape[:2]

@dataclass
class PointBatch:
    points: np.ndarray
    colors: np.ndarray = field(default=None)
    batch_index: int = 0

@dataclass
class RadarSample:
    range_bins: np.ndarray
    timestamp: float
    sample_index: int = 0

@runtime_checkable
class ScanAdapter(Protocol):
    async def get_rgb_frames(self, mission_id: str) -> AsyncIterator[Frame]: ...
    async def get_lidar_points(self, mission_id: str) -> AsyncIterator[PointBatch]: ...
    def get_total_frames(self) -> int: ...

@runtime_checkable
class DetectionAdapter(Protocol):
    async def get_thermal_frames(self, mission_id: str) -> AsyncIterator[Frame]: ...
    async def get_radar_returns(self, mission_id: str) -> AsyncIterator[RadarSample]: ...
    def get_survivor_ground_truth(self) -> list[dict]: ...

class MockScanAdapter:
    def __init__(self, num_frames=60, fps=5.0, num_lidar_batches=20, points_per_batch=5000):
        self.num_frames = num_frames
        self.fps = fps
        self.frame_delay = 1.0 / fps
        self.num_lidar_batches = num_lidar_batches
        self.points_per_batch = points_per_batch

    def get_total_frames(self): return self.num_frames

    async def get_rgb_frames(self, mission_id):
        for i in range(self.num_frames):
            h, w = 480, 640
            rng = np.random.RandomState(i * 42 + 7)
            noise = rng.randint(55, 125, (h, w), dtype=np.uint8)
            frame_data = np.zeros((h, w, 3), dtype=np.uint8)
            frame_data[:,:,0] = noise
            frame_data[:,:,1] = (noise * 0.82).astype(np.uint8)
            frame_data[:,:,2] = (noise * 0.65).astype(np.uint8)
            for _ in range(3):
                cx, cy = rng.randint(100,w-100), rng.randint(100,h-100)
                sw, sh = rng.randint(60,200), rng.randint(30,80)
                frame_data[max(0,cy-sh):cy+sh, max(0,cx-sw):cx+sw] = \
                    (frame_data[max(0,cy-sh):cy+sh, max(0,cx-sw):cx+sw] * 0.4).astype(np.uint8)
            yield Frame(index=i, data=frame_data, timestamp=i*self.frame_delay)
            await asyncio.sleep(self.frame_delay * 0.15)

    async def get_lidar_points(self, mission_id):
        for batch_idx in range(self.num_lidar_batches):
            rng = np.random.RandomState(batch_idx * 137 + 11)
            n = self.points_per_batch
            x = rng.uniform(0, 100, n).astype(np.float32)
            y = rng.uniform(0, 100, n).astype(np.float32)
            z = (2.0*np.sin(x*0.08)*np.cos(y*0.06)).astype(np.float32)
            d1 = np.sqrt((x-48)**2+(y-52)**2)
            z += (9.0*np.exp(-d1**2/350)).astype(np.float32)
            d2 = np.sqrt((x-22)**2+(y-68)**2)
            z += (6.0*np.exp(-d2**2/180)).astype(np.float32)
            d3 = np.sqrt((x-70)**2+(y-30)**2)
            z += (4.0*np.exp(-d3**2/120)).astype(np.float32)
            z += rng.uniform(-0.4, 0.4, n).astype(np.float32)
            points = np.stack([x, y, z], axis=1)
            z_norm = np.clip((z-z.min())/(z.max()-z.min()+1e-6), 0, 1)
            colors = np.zeros((n,3), dtype=np.uint8)
            colors[:,0] = (95+110*z_norm).astype(np.uint8)
            colors[:,1] = (80+85*z_norm).astype(np.uint8)
            colors[:,2] = (55+70*z_norm).astype(np.uint8)
            yield PointBatch(points=points, colors=colors, batch_index=batch_idx)
            await asyncio.sleep(0.3)

class MockDetectionAdapter:
    SURVIVOR_PROFILES = [
        {"id":"SRV-01","x":45,"y":50,"depth":2.3,"breathing_freq":0.23,"amplitude":0.7,"label":"Pinned under slab cavity"},
        {"id":"SRV-02","x":24,"y":65,"depth":1.1,"breathing_freq":0.28,"amplitude":0.85,"label":"Debris edge, partially visible"},
        {"id":"SRV-03","x":72,"y":32,"depth":3.4,"breathing_freq":0.19,"amplitude":0.5,"label":"Deep void, weak signal"},
    ]

    def __init__(self, num_thermal_frames=30, radar_duration_s=30.0, radar_sample_rate=20.0, num_range_bins=128):
        self.num_thermal_frames = num_thermal_frames
        self.radar_duration_s = radar_duration_s
        self.radar_sample_rate = radar_sample_rate
        self.num_range_bins = num_range_bins

    def get_survivor_ground_truth(self): return list(self.SURVIVOR_PROFILES)

    async def get_thermal_frames(self, mission_id):
        for i in range(self.num_thermal_frames):
            h, w = 480, 640
            rng = np.random.RandomState(i*53+7)
            bg_temp = rng.uniform(18, 25, (h, w)).astype(np.float64)
            for _ in range(2):
                ex, ey = rng.randint(50,w-50), rng.randint(50,h-50)
                ed = np.sqrt((np.arange(w)[None,:]-ex)**2+(np.arange(h)[:,None]-ey)**2)
                bg_temp += 8*np.exp(-ed**2/3000)
            for surv in self.SURVIVOR_PROFILES:
                cx = int(surv["x"]/100*w)
                cy = int(surv["y"]/100*h)
                dist = np.sqrt((np.arange(w)[None,:]-cx)**2+(np.arange(h)[:,None]-cy)**2)
                att = np.exp(-surv["depth"]*0.35)
                bm = 1.0+0.03*np.sin(2*np.pi*surv["breathing_freq"]*i*0.5)
                bg_temp += 37*att*bm*np.exp(-dist**2/(700+300*att))
            t_norm = np.clip((bg_temp-14)/32, 0, 1)
            fd = np.zeros((h,w,3), dtype=np.uint8)
            fd[:,:,0] = (t_norm*255).astype(np.uint8)
            fd[:,:,1] = (t_norm*(1-t_norm)*4*220).astype(np.uint8)
            fd[:,:,2] = ((1-t_norm)*200).astype(np.uint8)
            yield Frame(index=i, data=fd, timestamp=i*0.5)
            await asyncio.sleep(0.15)

    async def get_radar_returns(self, mission_id):
        num_samples = int(self.radar_sample_rate * self.radar_duration_s)
        static_rng = np.random.RandomState(42)
        static_clutter = static_rng.uniform(0.1, 0.85, self.num_range_bins).astype(np.float32)
        for t_idx in range(num_samples):
            t = t_idx / self.radar_sample_rate
            sample_rng = np.random.RandomState(t_idx*31+997)
            noise = sample_rng.normal(0, 0.04, self.num_range_bins).astype(np.float32)
            rb = static_clutter.copy() + noise
            for surv in self.SURVIVOR_PROFILES:
                bi = min(int(surv["depth"]/10.0*self.num_range_bins), self.num_range_bins-1)
                breathing = surv["amplitude"]*np.sin(2*np.pi*surv["breathing_freq"]*t)
                cardiac = 0.12*np.sin(2*np.pi*1.15*t)
                for off in range(-2,3):
                    idx = bi+off
                    if 0<=idx<self.num_range_bins:
                        rb[idx] += (breathing+cardiac)*np.exp(-off**2/1.8)
            yield RadarSample(range_bins=rb, timestamp=t, sample_index=t_idx)
            await asyncio.sleep(1.0/self.radar_sample_rate*0.05)

class LiveScanAdapter:
    async def get_rgb_frames(self, m): raise NotImplementedError("Connect real drone")
    async def get_lidar_points(self, m): raise NotImplementedError("Connect real LiDAR")
    def get_total_frames(self): raise NotImplementedError

class LiveDetectionAdapter:
    async def get_thermal_frames(self, m): raise NotImplementedError("Connect FLIR")
    async def get_radar_returns(self, m): raise NotImplementedError("Connect UWB")
    def get_survivor_ground_truth(self): return []

print("✅ Adapters loaded")

## 4. A* Pathfinder + Hazard Grid

In [ ]:
%%writefile pathfinder.py
"""Debris-Scan AI — A* Route Planning + Hazard Grid Generator"""
import numpy as np
import heapq
from dataclasses import dataclass, field as dc_field

@dataclass(order=True)
class _Node:
    f_cost: float
    position: tuple = dc_field(compare=False)
    g_cost: float = dc_field(compare=False, default=0.0)
    parent: '_Node | None' = dc_field(default=None, compare=False, repr=False)

class AStarPathfinder:
    DIRECTIONS = [(0,1),(1,0),(0,-1),(-1,0),(1,1),(1,-1),(-1,1),(-1,-1)]

    def __init__(self, grid_size=100):
        self.grid_size = grid_size

    def plan(self, start, goal, hazard_grid, mode="safest"):
        hw = 5.0 if mode=="safest" else 1.2
        sx = max(0,min(start[0],self.grid_size-1))
        sy = max(0,min(start[1],self.grid_size-1))
        gx = max(0,min(goal[0],self.grid_size-1))
        gy = max(0,min(goal[1],self.grid_size-1))
        s, g = (sx,sy), (gx,gy)
        if s==g: return self._trivial(s, hazard_grid)
        ops = []
        heapq.heappush(ops, _Node(0.0,s,0.0))
        gc = {s:0.0}; vis = set(); it = 0; mx = self.grid_size**2*4
        while ops and it < mx:
            it += 1; cur = heapq.heappop(ops)
            if cur.position==g: return self._recon(cur, hazard_grid, mode)
            if cur.position in vis: continue
            vis.add(cur.position)
            for dx,dy in self.DIRECTIONS:
                nx,ny = cur.position[0]+dx, cur.position[1]+dy
                if not(0<=nx<self.grid_size and 0<=ny<self.grid_size): continue
                if (nx,ny) in vis: continue
                ch = float(hazard_grid[ny,nx])
                if ch > 0.92: continue
                sd = np.sqrt(dx**2+dy**2)
                ec = sd*(1.0+hw*ch)
                ng = cur.g_cost+ec
                if (nx,ny) in gc and ng>=gc[(nx,ny)]: continue
                gc[(nx,ny)] = ng
                h = np.sqrt((nx-gx)**2+(ny-gy)**2)
                heapq.heappush(ops, _Node(ng+h,(nx,ny),ng,cur))
        d = np.sqrt((g[0]-s[0])**2+(g[1]-s[1])**2)
        return {"waypoints":[{"x":float(s[0]),"y":float(s[1]),"z":0,"hazard_cost":0},{"x":float(g[0]),"y":float(g[1]),"z":0,"hazard_cost":0}],"total_distance_m":round(d,1),"estimated_time_min":round(d/1.5/60,1),"max_hazard_encountered":0,"hazard_zone_crossings":0,"mode":mode,"note":"FALLBACK"}

    def _recon(self, node, hg, mode):
        wps=[]; cur=node
        while cur:
            x,y=cur.position; h=float(hg[y,x])
            wps.append({"x":float(x),"y":float(y),"z":0.0,"hazard_cost":round(h,4)})
            cur=cur.parent
        wps.reverse()
        td=sum(np.sqrt((wps[i]["x"]-wps[i-1]["x"])**2+(wps[i]["y"]-wps[i-1]["y"])**2) for i in range(1,len(wps)))
        mh=max(w["hazard_cost"] for w in wps)
        cr=sum(1 for w in wps if w["hazard_cost"]>0.3)
        ah=np.mean([w["hazard_cost"] for w in wps])
        es=1.5*(1.0-0.5*ah); tm=(td/max(es,0.3))/60
        if len(wps)>60:
            st=len(wps)//60; s=[wps[0]]
            for i in range(st,len(wps)-1,st): s.append(wps[i])
            s.append(wps[-1]); wps=s
        return {"waypoints":wps,"total_distance_m":round(td,1),"estimated_time_min":round(tm,1),"max_hazard_encountered":round(mh,3),"hazard_zone_crossings":cr,"mode":mode}

    def _trivial(self, pos, hg):
        h=float(hg[pos[1],pos[0]])
        return {"waypoints":[{"x":float(pos[0]),"y":float(pos[1]),"z":0,"hazard_cost":round(h,4)}],"total_distance_m":0,"estimated_time_min":0,"max_hazard_encountered":round(h,3),"hazard_zone_crossings":0,"mode":"safest"}

def generate_hazard_grid(point_cloud, grid_size=100):
    hazard = np.zeros((grid_size, grid_size), dtype=np.float32)
    if point_cloud is None or len(point_cloud)==0: return hazard
    x,y,z = point_cloud[:,0], point_cloud[:,1], point_cloud[:,2]
    xi = np.clip((x/100*grid_size).astype(int), 0, grid_size-1)
    yi = np.clip((y/100*grid_size).astype(int), 0, grid_size-1)
    for gx in range(grid_size):
        for gy in range(grid_size):
            mask = (xi==gx)&(yi==gy); cz = z[mask]
            if len(cz)<3: hazard[gy,gx]=0.6; continue
            zv = np.var(cz); zr = cz.max()-cz.min()
            hazard[gy,gx] = max(np.clip(zv/4.0,0,0.7), np.clip(zr/8.0,0,0.8))
    from scipy.ndimage import gaussian_filter
    hazard = gaussian_filter(hazard, sigma=1.2)
    hazard[45:55,42:52] = np.maximum(hazard[45:55,42:52], 0.85)
    hazard[28:35,60:70] = np.maximum(hazard[28:35,60:70], 0.75)
    return np.clip(hazard, 0.0, 1.0)

print("✅ Pathfinder loaded")

## 5. Multi-Modal Fusion Engine

In [ ]:
%%writefile fusion.py
"""Debris-Scan AI — Multi-Modal Fusion Engine"""
import numpy as np
from models import Detection, FusionResult

class MultiModalFusion:
    def __init__(self, w_thermal=0.35, w_radar=0.30, w_geom=0.25, w_hazard=0.10, spatial_merge_radius=8.0):
        self.w_thermal=w_thermal; self.w_radar=w_radar; self.w_geom=w_geom
        self.w_hazard=w_hazard; self.spatial_merge_radius=spatial_merge_radius

    def fuse(self, thermal_detections, radar_detections, point_cloud=None, hazard_grid=None, grid_size=100):
        all_d = [{"det":d,"source":"thermal"} for d in thermal_detections]
        all_d += [{"det":d,"source":"radar"} for d in radar_detections]
        if not all_d: return []
        clusters = self._cluster(all_d)
        results = [self._score(f"FUS-{i+1:02d}",c,point_cloud,hazard_grid,grid_size) for i,c in enumerate(clusters)]
        results.sort(key=lambda r: r.total_score, reverse=True)
        return results

    def _cluster(self, all_d):
        used=[False]*len(all_d); clusters=[]
        for i,a in enumerate(all_d):
            if used[i]: continue
            cl=[a]; used[i]=True
            for j,b in enumerate(all_d):
                if used[j]: continue
                d=np.sqrt((a["det"].x-b["det"].x)**2+(a["det"].y-b["det"].y)**2)
                if d<self.spatial_merge_radius: cl.append(b); used[j]=True
            clusters.append(cl)
        return clusters

    def _score(self, cid, cluster, pc, hg, gs):
        cx=np.mean([i["det"].x for i in cluster])
        cy=np.mean([i["det"].y for i in cluster])
        cz=np.mean([i["det"].z for i in cluster])
        th=[i for i in cluster if i["source"]=="thermal"]
        rd=[i for i in cluster if i["source"]=="radar"]
        ts=np.mean([i["det"].confidence for i in th]) if th else 0.0
        rs=np.mean([i["det"].confidence for i in rd]) if rd else 0.0
        ss=self._structural(cx,cy,pc,gs)
        hp=0.0
        if hg is not None:
            gx=int(np.clip(cx/100*gs,0,gs-1)); gy=int(np.clip(cy/100*gs,0,gs-1))
            hp=float(hg[gy,gx])
        total=self.w_thermal*ts+self.w_radar*rs+self.w_geom*ss-self.w_hazard*hp
        total=float(np.clip(total,0,1))
        if th and rd: total=min(1.0,total+0.08)
        return FusionResult(id=cid,x=round(float(cx),2),y=round(float(cy),2),z=round(float(cz),2),total_score=round(total,4),thermal_contribution=round(float(ts),4),radar_contribution=round(float(rs),4),structural_contribution=round(float(ss),4),hazard_penalty=round(float(hp),4))

    def _structural(self, x, y, pc, gs):
        if pc is None or len(pc)==0: return 0.5
        d=np.sqrt((pc[:,0]-x)**2+(pc[:,1]-y)**2)
        near=pc[d<5.0]
        if len(near)<5: return 0.85
        zv=np.var(near[:,2]); zr=near[:,2].max()-near[:,2].min()
        return float(np.clip(np.clip(zv/3.0,0,0.5)+np.clip(zr/10.0,0,0.5),0,1))

print("✅ Fusion engine loaded")

## 6. Pipeline Stage Implementations (All 8 Stages)

In [ ]:
%%writefile stages.py
"""Debris-Scan AI — 8-Stage Pipeline Implementations"""
import numpy as np, asyncio, base64
from datetime import datetime, timezone
from adapters import MockScanAdapter, MockDetectionAdapter, Frame, RadarSample
from models import Detection, FusionResult, RouteResult, Waypoint, MissionReport
from fusion import MultiModalFusion
from pathfinder import AStarPathfinder, generate_hazard_grid

async def stage_scan(mission_id, scan_adapter, progress_cb=None):
    fc=0; tf=scan_adapter.get_total_frames(); kf=[]
    async for frame in scan_adapter.get_rgb_frames(mission_id):
        fc+=1
        if frame.index%10==0: kf.append(frame)
        if progress_cb: await progress_cb(fc/tf,{"frames_captured":fc,"total_frames":tf,"current_frame_index":frame.index})
    ap=[]; ac=[]; bc=0
    async for batch in scan_adapter.get_lidar_points(mission_id):
        ap.append(batch.points)
        if batch.colors is not None: ac.append(batch.colors)
        bc+=1
    pc = np.concatenate(ap,axis=0) if ap else np.zeros((0,3))
    cl = np.concatenate(ac,axis=0) if ac else None
    return {"frames_captured":fc,"keyframes":kf,"point_cloud":pc,"point_cloud_colors":cl,"lidar_batches":bc,"total_points":len(pc)}

async def stage_reconstruct(mission_id, scan_data, progress_cb=None, use_real_model=False):
    pc=scan_data["point_cloud"]; cl=scan_data.get("point_cloud_colors")
    if progress_cb: await progress_cb(0.1,{"status":"Processing LiDAR point cloud"})
    if not use_real_model:
        await asyncio.sleep(0.5)
        rng=np.random.RandomState(2026); n=10000
        ex=rng.uniform(0,100,n).astype(np.float32); ey=rng.uniform(0,100,n).astype(np.float32)
        ez=(2.0*np.sin(ex*0.08)*np.cos(ey*0.06)).astype(np.float32)
        ez+=(9.0*np.exp(-((ex-48)**2+(ey-52)**2)/350)).astype(np.float32)
        ez+=(6.0*np.exp(-((ex-22)**2+(ey-68)**2)/180)).astype(np.float32)
        ez+=(4.0*np.exp(-((ex-70)**2+(ey-30)**2)/120)).astype(np.float32)
        ez+=rng.uniform(-0.8,0.8,n).astype(np.float32)
        ep=np.stack([ex,ey,ez],axis=1); pc=np.concatenate([pc,ep],axis=0)
        if cl is not None:
            zn=np.clip((ez-ez.min())/(ez.max()-ez.min()+1e-6),0,1)
            ec=np.zeros((n,3),dtype=np.uint8)
            ec[:,0]=(90+120*zn).astype(np.uint8); ec[:,1]=(75+95*zn).astype(np.uint8); ec[:,2]=(50+80*zn).astype(np.uint8)
            cl=np.concatenate([cl,ec],axis=0)
        if progress_cb: await progress_cb(0.6,{"status":"Fusing LiDAR + depth-estimated points","total_points":len(pc)})
    await asyncio.sleep(0.3)
    pb=base64.b64encode(pc.astype(np.float32).tobytes()).decode("ascii")
    cb=base64.b64encode(cl.astype(np.uint8).tobytes()).decode("ascii") if cl is not None else None
    if progress_cb: await progress_cb(1.0,{"status":"Reconstruction complete"})
    return {"point_cloud":pc,"point_cloud_colors":cl,"point_cloud_b64":pb,"color_b64":cb,"total_points":len(pc),"bounds":{"min":pc.min(axis=0).tolist(),"max":pc.max(axis=0).tolist()},"method":"mock_lidar_depth_fusion"}

async def stage_detect(mission_id, detection_adapter, progress_cb=None):
    thermal_dets=[]; fc=0
    if progress_cb: await progress_cb(0.05,{"status":"Running thermal sweep"})
    async for frame in detection_adapter.get_thermal_frames(mission_id):
        fc+=1
        tm=frame.data[:,:,0].astype(float)/255.0; h,w=tm.shape
        from scipy.ndimage import maximum_filter, label
        lm=maximum_filter(tm,size=40); peaks=(tm==lm)&(tm>0.55)
        labeled,nf=label(peaks)
        for fid in range(1,min(nf+1,6)):
            ys,xs=np.where(labeled==fid)
            if len(xs)==0: continue
            cx,cy=float(np.mean(xs)/w*100),float(np.mean(ys)/h*100)
            pv=float(tm[ys,xs].max())
            thermal_dets.append(Detection(id=f"TH-{frame.index:02d}-{fid}",source="thermal",x=round(cx,2),y=round(cy,2),confidence=round(pv,3),label="thermal_hotspot",metadata={"frame":frame.index}))
        if progress_cb: await progress_cb(0.05+0.4*fc/30,{"status":f"Thermal frame {fc}/30","thermal_candidates":len(thermal_dets)})
    if progress_cb: await progress_cb(0.5,{"status":"Processing UWB radar"})
    radar_dets = await _process_radar(mission_id, detection_adapter)
    td = _dedup(thermal_dets,6.0)
    if progress_cb: await progress_cb(1.0,{"status":"Detection complete","thermal_candidates":len(td),"radar_candidates":len(radar_dets)})
    return {"thermal_detections":td,"radar_detections":radar_dets,"thermal_frames_processed":fc,"radar_samples_processed":int(detection_adapter.radar_duration_s*detection_adapter.radar_sample_rate)}

async def _process_radar(mission_id, adapter):
    samples=[]
    async for s in adapter.get_radar_returns(mission_id): samples.append(s.range_bins)
    if not samples: return []
    rm=np.array(samples); ns,nb=rm.shape; sr=adapter.radar_sample_rate
    cl=np.mean(rm,axis=0); res=rm-cl; bv=np.var(res,axis=0)
    vt=np.percentile(bv,85); cands=np.where(bv>vt)[0]
    dets=[]; dc=0
    for bi in cands:
        sig=res[:,bi]; fv=np.abs(np.fft.rfft(sig)); fr=np.fft.rfftfreq(ns,d=1.0/sr)
        bm=(fr>=0.15)&(fr<=0.50)
        if not np.any(bm): continue
        bf=fv[bm]; bfr=fr[bm]
        if len(bf)==0: continue
        pi=np.argmax(bf); pp=bf[pi]; pf=bfr[pi]
        nf=np.median(fv[1:]); snr=pp/(nf+1e-8)
        if snr>3.0:
            dc+=1; depth=bi/nb*10.0; conf=float(np.clip(snr/15.0,0.3,0.95))
            dets.append(Detection(id=f"UWB-{dc:02d}",source="radar",x=round(bi/nb*100,2),y=round(50+(bi%7-3)*8,2),z=round(depth,2),confidence=round(conf,3),label="breathing_signature",metadata={"breathing_freq_hz":round(float(pf),3),"breathing_bpm":round(float(pf*60),1),"snr_db":round(float(10*np.log10(snr)),1),"range_bin":int(bi),"depth_m":round(depth,2),"method":"variance_threshold_fft"}))
    return dets

def _dedup(dets, radius=6.0):
    if not dets: return []
    sd=sorted(dets,key=lambda d:d.confidence,reverse=True); kept=[]; used=set()
    for d in sd:
        if d.id in used: continue
        for o in sd:
            if o.id in used or o.id==d.id: continue
            if np.sqrt((d.x-o.x)**2+(d.y-o.y)**2)<radius: used.add(o.id)
        kept.append(d); used.add(d.id)
    return kept

async def stage_fuse(mission_id, detect_data, reconstruct_data, progress_cb=None):
    if progress_cb: await progress_cb(0.1,{"status":"Computing multi-modal fusion scores"})
    f=MultiModalFusion(); pc=reconstruct_data.get("point_cloud")
    hg=generate_hazard_grid(pc)
    results=f.fuse(detect_data["thermal_detections"],detect_data["radar_detections"],pc,hg)
    if progress_cb: await progress_cb(1.0,{"status":"Fusion complete","candidates":len(results),"top_score":results[0].total_score if results else 0})
    return {"fusion_results":results,"hazard_grid":hg,"total_candidates":len(results)}

async def stage_revisit(mission_id, fuse_data, progress_cb=None):
    results=fuse_data["fusion_results"]; confirmed=[]; rejected=[]
    for i,c in enumerate(results):
        if progress_cb: await progress_cb((i+1)/len(results),{"status":f"Revisiting {c.id}","position":{"x":c.x,"y":c.y}})
        await asyncio.sleep(0.4)
        if c.total_score>0.35: c.confirmed=True; confirmed.append(c)
        else: rejected.append(c)
    return {"confirmed":confirmed,"rejected":rejected,"confirmed_count":len(confirmed),"rejected_count":len(rejected)}

async def stage_hazard(mission_id, reconstruct_data, fuse_data, progress_cb=None):
    if progress_cb: await progress_cb(0.1,{"status":"Voxelizing for hazard analysis"})
    hg=fuse_data.get("hazard_grid")
    if hg is None: hg=generate_hazard_grid(reconstruct_data.get("point_cloud"))
    await asyncio.sleep(0.5)
    gs=hg.shape[0]; summary={"stable":0,"loose_rubble":0,"steep":0,"void":0,"impassable":0}
    for gy in range(gs):
        for gx in range(gs):
            c=float(hg[gy,gx])
            if c>0.9: summary["impassable"]+=1
            elif c>0.6: summary["steep"]+=1
            elif c>0.35: summary["loose_rubble"]+=1
            else: summary["stable"]+=1
    hb=base64.b64encode(hg.astype(np.float32).tobytes()).decode("ascii")
    if progress_cb: await progress_cb(1.0,{"status":"Hazard mapping complete","summary":summary})
    return {"hazard_grid":hg,"hazard_grid_b64":hb,"grid_size":gs,"summary":summary}

async def stage_route(mission_id, hazard_data, revisit_data, progress_cb=None):
    hg=hazard_data["hazard_grid"]; confirmed=revisit_data["confirmed"]; gs=hg.shape[0]
    pf=AStarPathfinder(grid_size=gs); entry=(5,95); routes=[]
    for i,c in enumerate(confirmed):
        if progress_cb: await progress_cb((i+0.5)/max(len(confirmed),1),{"status":f"Planning routes to {c.id}"})
        goal=(int(np.clip(c.x,0,gs-1)),int(np.clip(c.y,0,gs-1)))
        for mode in ["safest","fastest"]:
            r=pf.plan(entry,goal,hg,mode=mode)
            routes.append(RouteResult(target_id=c.id,mode=mode,waypoints=[Waypoint(**w) for w in r["waypoints"]],total_distance_m=r["total_distance_m"],estimated_time_min=r["estimated_time_min"],max_hazard_encountered=r["max_hazard_encountered"],hazard_zone_crossings=r["hazard_zone_crossings"]))
        await asyncio.sleep(0.2)
    if progress_cb: await progress_cb(1.0,{"status":"All routes computed","total_routes":len(routes)})
    return {"routes":routes,"entry_point":list(entry),"total_routes":len(routes)}

async def stage_report(mission_id, mission_data, progress_cb=None):
    if progress_cb: await progress_cb(0.5,{"status":"Generating mission report"})
    scan=mission_data.get("scan",{}); recon=mission_data.get("reconstruct",{})
    fuse_d=mission_data.get("fuse",{}); rev=mission_data.get("revisit",{})
    haz=mission_data.get("hazard",{}); rt=mission_data.get("route",{})
    report=MissionReport(mission_id=mission_id,site_name=mission_data.get("site_name","Unknown"),adapter_mode="mock",total_area_scanned_m2=10000.0,scan_duration_s=scan.get("frames_captured",60)*0.2,total_points_captured=recon.get("total_points",0),candidates_detected=fuse_d.get("total_candidates",0),confirmed_survivors=rev.get("confirmed_count",0),fusion_results=rev.get("confirmed",[]),routes=rt.get("routes",[]),hazard_summary=haz.get("summary",{}))
    rj=report.model_dump(mode="json")
    if progress_cb: await progress_cb(1.0,{"status":"Report generated"})
    return {"report":rj}

print("✅ All 8 stages loaded")

## 7. Mission Orchestrator

In [ ]:
%%writefile orchestrator.py
"""Debris-Scan AI — Mission Orchestrator"""
import asyncio, traceback
from datetime import datetime, timezone
from typing import Callable, Awaitable
from models import Mission, StageEvent, StageEnum, StatusEnum, STAGE_ORDER, FusionResult, RouteResult
from adapters import MockScanAdapter, MockDetectionAdapter
from stages import stage_scan, stage_reconstruct, stage_detect, stage_fuse, stage_revisit, stage_hazard, stage_route, stage_report

EventCallback = Callable[[StageEvent], Awaitable[None]]

class MissionOrchestrator:
    def __init__(self):
        self.missions = {}
        self.mission_data = {}
        self._locks = {}
        self.scan_adapter = MockScanAdapter(num_frames=60, fps=5.0)
        self.detection_adapter = MockDetectionAdapter(num_thermal_frames=30, radar_duration_s=30.0)

    def create_mission(self, site_name, boundary_coords=None):
        m = Mission(site_name=site_name, boundary_coords=boundary_coords or [[28.6139,77.2090],[28.6145,77.2090],[28.6145,77.2096],[28.6139,77.2096]], adapter_mode="mock")
        for s in STAGE_ORDER: m.stage_statuses[s.value] = StatusEnum.PENDING.value
        self.missions[m.id]=m; self.mission_data[m.id]={"site_name":site_name}; self._locks[m.id]=asyncio.Lock()
        return m

    def get_mission(self, mid): return self.missions.get(mid)

    def get_mission_snapshot(self, mid):
        m=self.missions.get(mid)
        if not m: return None
        data=self.mission_data.get(mid,{}); snap=m.model_dump(mode="json"); arts={}
        if "reconstruct" in data:
            r=data["reconstruct"]
            arts["reconstruct"]={"total_points":r.get("total_points",0),"point_cloud_b64":r.get("point_cloud_b64"),"color_b64":r.get("color_b64"),"bounds":r.get("bounds"),"method":r.get("method")}
        if "fuse" in data:
            f=data["fuse"]
            arts["fuse"]={"total_candidates":f.get("total_candidates",0),"fusion_results":[r.model_dump(mode="json") if hasattr(r,"model_dump") else r for r in f.get("fusion_results",[])]}
        if "revisit" in data:
            rv=data["revisit"]
            arts["revisit"]={"confirmed_count":rv.get("confirmed_count",0),"confirmed":[r.model_dump(mode="json") if hasattr(r,"model_dump") else r for r in rv.get("confirmed",[])]}
        if "hazard" in data:
            h=data["hazard"]
            arts["hazard"]={"hazard_grid_b64":h.get("hazard_grid_b64"),"grid_size":h.get("grid_size"),"summary":h.get("summary")}
        if "route" in data:
            rt=data["route"]
            arts["route"]={"total_routes":rt.get("total_routes",0),"entry_point":rt.get("entry_point"),"routes":[r.model_dump(mode="json") if hasattr(r,"model_dump") else r for r in rt.get("routes",[])]}
        if "report" in data: arts["report"]=data["report"].get("report")
        snap["artifacts"]=arts
        return snap

    async def run_mission(self, mid, broadcast):
        m=self.missions.get(mid)
        if not m: raise ValueError(f"Mission {mid} not found")
        async with self._locks[mid]:
            m.started_at=datetime.now(timezone.utc); data=self.mission_data[mid]
            stage_fns=[
                (StageEnum.SCAN, self._run_scan), (StageEnum.RECONSTRUCT, self._run_recon),
                (StageEnum.DETECT, self._run_detect), (StageEnum.FUSE, self._run_fuse),
                (StageEnum.REVISIT, self._run_revisit), (StageEnum.HAZARD, self._run_hazard),
                (StageEnum.ROUTE, self._run_route), (StageEnum.REPORT, self._run_report),
            ]
            for se, fn in stage_fns:
                try:
                    m.current_stage=se; m.stage_statuses[se.value]=StatusEnum.STARTED.value
                    await broadcast(StageEvent(mission_id=mid,stage=se,status=StatusEnum.STARTED))
                    async def pcb(p,pl,_s=se): await broadcast(StageEvent(mission_id=mid,stage=_s,status=StatusEnum.PROGRESS,progress=round(p,3),payload=pl))
                    result=await fn(mid,data,pcb); data[se.value]=result
                    cp=self._serialize(se,result)
                    m.stage_statuses[se.value]=StatusEnum.COMPLETE.value
                    await broadcast(StageEvent(mission_id=mid,stage=se,status=StatusEnum.COMPLETE,progress=1.0,payload=cp))
                except Exception as e:
                    m.stage_statuses[se.value]=StatusEnum.FAILED.value
                    await broadcast(StageEvent(mission_id=mid,stage=se,status=StatusEnum.FAILED,payload={"error":str(e),"traceback":traceback.format_exc()}))
                    return
            m.completed_at=datetime.now(timezone.utc)

    async def _run_scan(self,mid,d,pcb): return await stage_scan(mid,self.scan_adapter,pcb)
    async def _run_recon(self,mid,d,pcb): return await stage_reconstruct(mid,d.get("scan",{}),pcb)
    async def _run_detect(self,mid,d,pcb): return await stage_detect(mid,self.detection_adapter,pcb)
    async def _run_fuse(self,mid,d,pcb): return await stage_fuse(mid,d.get("detect",{}),d.get("reconstruct",{}),pcb)
    async def _run_revisit(self,mid,d,pcb): return await stage_revisit(mid,d.get("fuse",{}),pcb)
    async def _run_hazard(self,mid,d,pcb): return await stage_hazard(mid,d.get("reconstruct",{}),d.get("fuse",{}),pcb)
    async def _run_route(self,mid,d,pcb): return await stage_route(mid,d.get("hazard",{}),d.get("revisit",{}),pcb)
    async def _run_report(self,mid,d,pcb): return await stage_report(mid,d,pcb)

    def _serialize(self, stage, result):
        if stage==StageEnum.SCAN: return {"frames_captured":result.get("frames_captured",0),"total_points":result.get("total_points",0),"lidar_batches":result.get("lidar_batches",0)}
        elif stage==StageEnum.RECONSTRUCT: return {"total_points":result.get("total_points",0),"point_cloud_b64":result.get("point_cloud_b64"),"color_b64":result.get("color_b64"),"bounds":result.get("bounds"),"method":result.get("method")}
        elif stage==StageEnum.DETECT:
            th=result.get("thermal_detections",[]); rd=result.get("radar_detections",[])
            return {"thermal_candidates":len(th),"radar_candidates":len(rd),"thermal_detections":[d.model_dump(mode="json") if hasattr(d,"model_dump") else d for d in th],"radar_detections":[d.model_dump(mode="json") if hasattr(d,"model_dump") else d for d in rd]}
        elif stage==StageEnum.FUSE:
            rs=result.get("fusion_results",[])
            return {"total_candidates":len(rs),"fusion_results":[r.model_dump(mode="json") if hasattr(r,"model_dump") else r for r in rs]}
        elif stage==StageEnum.REVISIT:
            c=result.get("confirmed",[])
            return {"confirmed_count":result.get("confirmed_count",0),"rejected_count":result.get("rejected_count",0),"confirmed":[r.model_dump(mode="json") if hasattr(r,"model_dump") else r for r in c]}
        elif stage==StageEnum.HAZARD: return {"hazard_grid_b64":result.get("hazard_grid_b64"),"grid_size":result.get("grid_size"),"summary":result.get("summary")}
        elif stage==StageEnum.ROUTE:
            rs=result.get("routes",[])
            return {"total_routes":len(rs),"entry_point":result.get("entry_point"),"routes":[r.model_dump(mode="json") if hasattr(r,"model_dump") else r for r in rs]}
        elif stage==StageEnum.REPORT: return result.get("report",{})
        return {}

print("✅ Orchestrator loaded")

## 8. FastAPI Server

In [ ]:
%%writefile server.py
"""Debris-Scan AI — FastAPI Server"""
import asyncio, json, logging
from contextlib import asynccontextmanager
from datetime import datetime, timezone
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from models import MissionCreate, StageEvent
from orchestrator import MissionOrchestrator

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("debris-scan")

class ConnectionManager:
    def __init__(self): self.connections = {}
    async def connect(self, mid, ws):
        await ws.accept()
        if mid not in self.connections: self.connections[mid]=[]
        self.connections[mid].append(ws)
        logger.info(f"WS connected: mission={mid}")
    def disconnect(self, mid, ws):
        if mid in self.connections: self.connections[mid]=[c for c in self.connections[mid] if c!=ws]
    async def broadcast(self, mid, event):
        if mid not in self.connections: return
        msg=event.model_dump(mode="json")
        msg["timestamp"]=msg["timestamp"].isoformat() if isinstance(msg["timestamp"],datetime) else str(msg["timestamp"])
        dead=[]
        for ws in self.connections[mid]:
            try: await ws.send_json(msg)
            except: dead.append(ws)
        for ws in dead: self.connections[mid]=[c for c in self.connections[mid] if c!=ws]

wm = ConnectionManager()
orch = MissionOrchestrator()

@asynccontextmanager
async def lifespan(app):
    logger.info("Debris-Scan AI backend starting"); yield; logger.info("Shutting down")

app = FastAPI(title="Debris-Scan AI", version="1.0.0", lifespan=lifespan)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.get("/")
async def root():
    return {"service":"Debris-Scan AI","version":"1.0.0","status":"operational","adapter_mode":"mock","missions_active":len(orch.missions)}

@app.post("/missions", status_code=201)
async def create_mission(req: MissionCreate):
    m=orch.create_mission(site_name=req.site_name, boundary_coords=req.boundary_coords)
    logger.info(f"Mission created: {m.id}"); return m.model_dump(mode="json")

@app.post("/missions/{mid}/start", status_code=202)
async def start_mission(mid: str):
    m=orch.get_mission(mid)
    if not m: raise HTTPException(404,"Mission not found")
    if m.started_at: raise HTTPException(409,"Already started")
    async def bfn(e): logger.info(f"[{e.stage.value}] {e.status.value} p={e.progress}"); await wm.broadcast(mid,e)
    asyncio.create_task(orch.run_mission(mid,bfn))
    return {"mission_id":mid,"status":"pipeline_started"}

@app.get("/missions/{mid}")
async def get_mission(mid: str):
    s=orch.get_mission_snapshot(mid)
    if not s: raise HTTPException(404,"Mission not found")
    return s

@app.get("/missions")
async def list_missions():
    return {"missions":[{"id":m.id,"site_name":m.site_name,"current_stage":m.current_stage.value,"created_at":m.created_at.isoformat(),"started_at":m.started_at.isoformat() if m.started_at else None,"completed_at":m.completed_at.isoformat() if m.completed_at else None} for m in orch.missions.values()]}

@app.websocket("/missions/{mid}/events")
async def mission_events(websocket: WebSocket, mid: str):
    m=orch.get_mission(mid)
    if not m: await websocket.close(code=4004,reason="Not found"); return
    await wm.connect(mid,websocket)
    try:
        snap=orch.get_mission_snapshot(mid)
        if snap: await websocket.send_json({"type":"snapshot","data":snap})
        while True:
            try:
                data=await asyncio.wait_for(websocket.receive_text(),timeout=30.0)
                msg=json.loads(data) if data else {}
                if msg.get("type")=="ping": await websocket.send_json({"type":"pong"})
            except asyncio.TimeoutError:
                try: await websocket.send_json({"type":"ping"})
                except: break
    except WebSocketDisconnect: pass
    except Exception as e: logger.error(f"WS error: {e}")
    finally: wm.disconnect(mid,websocket)

@app.get("/health")
async def health(): return {"status":"healthy","timestamp":datetime.now(timezone.utc).isoformat()}

print("✅ Server loaded")

## 9. 🚀 Start the Server with ngrok

This cell:
1. Starts the FastAPI server on port 8000
2. Creates an ngrok tunnel to expose it publicly
3. Prints the **public URL** — copy this into the frontend

> 💡 Get a free ngrok auth token at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) and paste it below.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

# ── Set your ngrok auth token here ──
NGROK_AUTH_TOKEN = ""  # <-- Paste your token from dashboard.ngrok.com

from pyngrok import ngrok, conf
import uvicorn
import threading

# Configure ngrok
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start uvicorn in a background thread
def run_server():
    uvicorn.run("server:app", host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time
time.sleep(3)  # Wait for server to start

# Open ngrok tunnel
public_url = ngrok.connect(8000, "http").public_url

print("\n" + "="*60)
print("🛰️  DEBRIS-SCAN AI BACKEND IS LIVE")
print("="*60)
print(f"\n  📡 Public URL:  {public_url}")
print(f"  📋 API Docs:    {public_url}/docs")
print(f"  🏥 Health:      {public_url}/health")
print(f"\n  ➡️  Paste this URL into the frontend's backend connector.")
print("="*60)
print("\n⚠️  Keep this notebook running while using the frontend.")
print("   The server will stop when you close this notebook.")